This is a file for prototyping using the models/dataloaders defined elsewhere. In addition, it contains the training loop but should NOT include the actual models themselves

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
# from torchvision import transforms
# from torchvision.transforms import RandAugment
import transformers

from transformers import get_cosine_schedule_with_warmup


import random

from tqdm import tqdm
import sys


# from google.colab import drive

torch.manual_seed(0)

# drive.mount('/content/drive', force_remount=True)


base_dir = "/content/drive/MyDrive/Junior/Second Semester/CS 4782 - DL/Final Project - HRM"
# sys.path.append(base_dir)
from Model.HRM_Model import HRM
from Model.HRM_Components import Encoder, HighLevel, LowLevel, Head
from Datasets.Sudoku_DataLoader import get_loaders

# sys.path.append(base_dir)
device = torch.device("cuda" if torch.cuda.is_available() else 
                        ("mps" if torch.backends.mps.is_available() else"cpu"))
# device = torch.device("meta")
print(F"Device set to {device}")

Device set to mps


In [6]:
train_size = 1024
test_size = 100
batch_size = 64
train_dataloader, test_dataloader = get_loaders(train_size, test_size, batch_size)


Map: 100%|██████████| 100/100 [00:00<00:00, 11442.96 examples/s]


In [7]:
#Define Model Hyperparameters
d_model = 512
M = 1
N = 2
T = 2
n_layers = 4
n_heads = 8
vocab_size = 10
lr = 1e-3
beta1 = .9
beta2 = .95
weight_decay = .1
num_epochs = 100
num_warmup_steps = 2000

high_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
low_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
head = Head(vocab_size=vocab_size, d_model=d_model)
encoder = Encoder(vocab_size=vocab_size, d_model=d_model, max_len=81)


HRM_model = HRM(low_level, high_level, encoder, head, M, N, T).to(device)

# Define the optimizer
optimizer = optim.AdamW(
    HRM_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay
)
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = num_warmup_steps

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
    num_cycles=0.5  # standard cosine
)

In [11]:
def train_hrm_deepsup(model, loader, optimizer, loss_fn, scheduler=None, num_epochs=10, vocab_size=10):
    """
    Train HRM with deep supervision for multiple epochs and report stats.

    model      : HRM model
    loader     : DataLoader with (input_seq, target_seq)
    optimizer  : optimizer
    loss_fn    : loss function (nn.CrossEntropyLoss)
    scheduler  : optional learning rate scheduler
    num_epochs : number of epochs to train
    vocab_size : size of discrete token vocabulary
    """
    model.train()
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    for epoch in range(num_epochs):
        correct_tokens = 0
        total_tokens = 0
        epoch_loss = 0

        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x = x.to(device)
            y = y.to(device)
            x_embed = model.encode(x)  # trainable encoder
            B, L, d = x_embed.shape
            z_H = torch.zeros(B, L, d, device=x.device)
            z_L = torch.zeros(B, L, d, device=x.device)
            x_flat = x.view(-1)
            y_flat = y.view(-1)
            mask = (x_flat != y_flat)
            targets = y_flat.clone()
            targets[~mask] = -100

            optimizer.zero_grad()
            total_loss_batch = torch.zeros((), device=device)
            losses = []

            for m in range(model.M):
                z_H = z_H.detach()
                z_L = z_L.detach()
                # Detach x_embed after first segment so encoder only trained once
                x_in = x_embed if m == 0 else x_embed.detach()
                z_H, z_L = model.step(z_H, z_L, x_in)

                # Segment-wise forward
                logits = model.head(z_H)  # (B, L, vocab_size)

                pred = logits.view(B * L, vocab_size)

                loss = loss_fn(pred, targets)
                losses.append(loss)
            total_loss_batch = torch.stack(losses).mean()

            total_loss_batch.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            epoch_loss += total_loss_batch.item()
            with torch.no_grad():
                # pred_labels = pred.argmax(dim=1)
                probs = torch.softmax(pred, dim=1)
                dist = torch.distributions.Categorical(probs=probs)
                pred_labels = dist.sample()
                correct_tokens += (pred_labels[mask] == y_flat[mask]).sum()
                total_tokens +=  mask.sum()
        avg_loss = epoch_loss / len(loader)
        acc = correct_tokens.item() / total_tokens.item()
        
        print(f"Epoch {epoch+1}: Avg Loss = {avg_loss:.4f}, Token Accuracy = {acc*100:.2f}%")

    return model
train_hrm_deepsup(HRM_model,train_dataloader, optimizer,nn.CrossEntropyLoss(ignore_index=-100), scheduler, num_epochs=num_epochs, vocab_size=vocab_size)
HRM_model.eval()

Number of trainable parameters: 25,270,794


Epoch 1/100: 100%|██████████| 16/16 [00:12<00:00,  1.24it/s]


Epoch 1: Avg Loss = 2.1563, Token Accuracy = 11.91%, Expected Accuracy = 11.58%


Epoch 2/100: 100%|██████████| 16/16 [00:11<00:00,  1.35it/s]


Epoch 2: Avg Loss = 2.1549, Token Accuracy = 12.15%, Expected Accuracy = 11.59%


...

Epoch 99/100: 100%|██████████| 16/16 [00:11<00:00,  1.40it/s]


Epoch 99: Avg Loss = 0.6632, Token Accuracy = 66.63%, Expected Accuracy = 51.52%


Epoch 100/100: 100%|██████████| 16/16 [00:11<00:00,  1.41it/s]

Epoch 100: Avg Loss = 0.6369, Token Accuracy = 67.63%, Expected Accuracy = 52.89%


HRM(
  (low_level): HighLevel(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (high_level): HighLevel(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizable

In [65]:
#Visualize a sudoku
data_iter = iter(test_dataloader)
x_batch, y_batch = next(data_iter)
x = x_batch[0]
y = y_batch[0]
# x,y = train_dataset[random.randint(0,train_size-1)]
x = torch.tensor(x, dtype=torch.long)
print("X: " + str(x))
pred = HRM_model.predict(x).to("cpu")
print("Prediction: " + str(pred))
print("Y: " + str(y))
cor_tok = (pred == y).sum()
print("Prediction Accuracy: " + str(cor_tok / 81))


X: tensor([0, 9, 0, 0, 0, 1, 2, 0, 0, 0, 3, 0, 0, 2, 8, 4, 0, 6, 0, 6, 0, 0, 0, 0,
        0, 8, 0, 0, 7, 0, 0, 0, 0, 1, 4, 0, 0, 2, 0, 0, 5, 0, 7, 0, 0, 0, 0, 3,
        0, 0, 0, 0, 0, 2, 0, 1, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 5, 0, 0, 0,
        0, 8, 7, 2, 0, 6, 0, 0, 4])
Prediction: tensor([[3, 9, 2, 3, 4, 1, 2, 1, 2, 9, 3, 2, 8, 2, 2, 4, 9, 6, 3, 6, 5, 2, 7, 9,
         1, 4, 4, 2, 3, 4, 1, 5, 7, 1, 4, 9, 4, 9, 5, 3, 5, 6, 3, 5, 7, 5, 7, 1,
         8, 9, 9, 5, 1, 9, 4, 1, 3, 9, 5, 5, 6, 7, 5, 7, 2, 1, 3, 3, 5, 4, 7, 7,
         7, 2, 3, 2, 8, 1, 4, 8, 4]])
Y: tensor([5, 9, 8, 4, 6, 1, 2, 7, 3, 7, 3, 1, 5, 2, 8, 4, 9, 6, 2, 6, 4, 3, 9, 7,
        5, 8, 1, 8, 7, 9, 6, 3, 2, 1, 4, 5, 4, 2, 6, 1, 5, 9, 7, 3, 8, 1, 5, 3,
        8, 7, 4, 9, 6, 2, 6, 1, 5, 9, 4, 3, 8, 2, 7, 3, 4, 2, 7, 8, 5, 6, 1, 9,
        9, 8, 7, 2, 1, 6, 3, 5, 4])
Prediction Accuracy: tensor(0.2346)


/var/folders/hg/y35ccm617xld9rm686z5l2q80000gn/T/ipykernel_7554/655470511.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long)


In [66]:
#Save the model
torch.save(HRM_model.state_dict(), "hrm_model.pt")